# Open-Meteo temperature extraction

This notebook is the cleaned version of the workflow used to create the
analysis-ready temperature file. Set the `OPEN_METEO_API_KEY` environment
variable before running it. Cell outputs and execution counts have been
removed for sharing.


### This script uses Open-Meteo API to extract the daily mean temperature (Tmean) using point locations of lsoa coordinates. The script uses the licensed (commercial) version of the API, but the free version can also be used though only limited API calls will be available in that case (read Open-Meteo documentation for further details). Open-Metero includes an option to extract the daily Tmean for the local timezone. To handle time zones robustly, the parameter 'auto' can be set in the API call. However, this does not always work correctly for all coordinates. To bypass the error, the timezone for each coordinate is first set manually, and only when the extraction fails using the manual timezone, the API call reverts to the 'auto' options using a try/exception error catching approach.

### Data:
####  Coordinates: centroid for each of 34,753 LSOAs defined using 2011 boundaries in England and Wales.


### Useful links

#### https://open-meteo.com/en/docs/historical-weather-api#latitude=-34.92,-27.47&longitude=138.6,153.03&start_date=2023-01-01&end_date=2023-12-31&hourly=&daily=temperature_2m_mean&timezone=auto
#### https://stackoverflow.com/questions/40700039/how-can-you-clone-a-conda-environment-into-the-root-environment
#### https://github.com/open-meteo/open-meteo/discussions/650
#### https://github.com/open-meteo/open-meteo/discussions/696
#### https://pypi.org/project/timezonefinder/

In [ ]:
import sys
print(sys.executable) 

In [ ]:
!{sys.executable} -m pip install pandas pyarrow requests timezonefinder


### Install required packages before loading

In [ ]:
import os
import requests
import pandas as pd
import pyarrow
from timezonefinder import TimezoneFinder


In [ ]:
modules = ["requests", "pandas", "pyarrow", "timezonefinder"]

for module in modules:
    try:
        __import__(module)
        print(f"{module} imported successfully")
    except ImportError as error:
        print(f"{module} is not installed: {error}")


### Set path to home directory, input csv file (ucdb_coordinates), output folder, and output parquet file

In [ ]:

print(os.getcwd()) 

In [ ]:
from pathlib import Path

# Run from the repository root or from the data-preparation folder.
repo_root = Path.cwd()
if not (repo_root / "data").exists():
    repo_root = repo_root.parent
if not (repo_root / "data" / "lsoa_coor.csv").exists():
    raise FileNotFoundError("Run this notebook from the UK-HW2026 repository")

input_coords = repo_root / "data" / "lsoa_coor.csv"
out_dir_parquet = repo_root / "download"
out_file_parquet = out_dir_parquet / "LSOA_daily_ERA5_2026m2s_t2m.parquet"
out_dir_parquet.mkdir(parents=True, exist_ok=True)

lsoa_df = pd.read_csv(input_coords)


### Read the input coordinates from the input .csv file. Rename columns and round the lon, lat to decimals

In [ ]:
lsoa_df = lsoa_df[['LSOA11CD', 'lat', 'lon']]
lsoa_df["lat"] = lsoa_df["lat"].astype(float)
lsoa_df["lon"] = lsoa_df["lon"].astype(float)
print(lsoa_df)

### Create a column of timezone in the above DF using TimezoneFinder() and a function to apply it row-wise on the DF

In [ ]:
tf = TimezoneFinder()

def get_timezone(row):
    return tf.timezone_at(lng=row['lon'], lat=row['lat'])  # Note: longitude first

### Apply function row-wise

In [ ]:
lsoa_df['timezone'] = lsoa_df.apply(get_timezone, axis=1)
lsoa_df

In [ ]:
# Check if any rows in the column 'timezone' have missing values
print(lsoa_df['timezone'].isnull().any())

# How many are missing
print(lsoa_df['timezone'].isnull().sum())

#To check which rows are missing
lsoa_df[lsoa_df['timezone'].isnull()]

### Define a function to extract the daily Tmean for the required time period. Use the API key and set the base_url to "https://customer-archive-api.open-meteo.com/v1/archive"
### Alternatively, set base_url to "https://archive-api.open-meteo.com/v1/archive" if using the free version.

### Within this parent function, use try and exception to check for errors if the timezone from the DF column returns an error when extracting the daily Tmean. Use the 'auto' option for the timezone in that case.

In [ ]:
def fetch_daily_mean_temp(lat, lon, timezone):
    #base_url = "https://archive-api.open-meteo.com/v1/archive"  # Use this for free version of Open-Meteo and comment the apikey parameter below
    base_url = "https://customer-archive-api.open-meteo.com/v1/archive"
    
    date_ranges = [
        ("2026-05-01", "2026-09-30")
    ]

    def make_request(tz, start, end):
        params = {
            "latitude": lat,
            "longitude": lon,
            "start_date": start,
            "end_date": end,            
            "daily": "temperature_2m_mean",
            #"models": "era5_land",
            "apikey": os.environ["OPEN_METEO_API_KEY"],
            "timezone": tz
        }
        response = requests.get(base_url, params=params)
        response.raise_for_status()
        data = response.json()
        if "daily" not in data or "temperature_2m_mean" not in data["daily"]:
            raise ValueError("Missing daily mean temperature in response")
        return data["daily"]
        
    all_times = []
    all_temps = []
    
    for start, end in date_ranges:
        try:
            result = make_request(timezone, start, end)
        except Exception as e:
            print(f"⚠️ Failed with timezone '{timezone}' at ({lat}, {lon}): {e}")
            print("Retrying with timezone='auto'...")
            try:
                result = make_request("auto", start, end)
            except Exception as e2:
                print(f"❌ Failed with 'auto' as well: {e2}")
                return None
        
        # combine 
        all_times.extend(result["time"])
        all_temps.extend(result["temperature_2m_mean"])
    
    return {"time": all_times, "temperature_2m_mean": all_temps}

### Now run the above function for each row (location) in the DF. The result of the extaction will be in a form of list, which can be flattened as individual rows and appended to the output DF. The columns retained are LSOA11CD, lat, lon, date and the temperature. But note that the output DF is not a Pandas DF and the conversion to Pandas DF is done separately in the block below

In [ ]:
%%time

flattened_rows = []

for _, row in lsoa_df.iterrows():
    print(f"\nCurrent location is: {row['LSOA11CD']}\n")
    daily_data = fetch_daily_mean_temp(row['lat'], row['lon'], row['timezone'])
    
    if daily_data:
        for date, temp in zip(daily_data['time'], daily_data['temperature_2m_mean']):
            flattened_rows.append({
                'LSOA11CD': row['LSOA11CD'],
                'lat': row['lat'],
                'lon': row['lon'],
                'date': date,
                # Column renamed to t2m
                't2m': temp
            })
    else:
        print(f"❌ No data for {row['LSOA11CD']}")


### Convert to Pandas DF

In [ ]:
lsoa_df_flat = pd.DataFrame(flattened_rows)
print(lsoa_df_flat)

### Check for missing temperature values

In [ ]:
print(lsoa_df_flat['t2m'].isnull().any())
# False

# How many are missing
lsoa_df_flat['t2m'].isnull().sum()


### Check min and max across all daily Tmean (i.e., all timesteps and all locations)

In [ ]:
lsoa_df_flat['t2m'].min()

In [ ]:
lsoa_df_flat['t2m'].max()

#### Save the large Pandas DF as a parquet file

In [ ]:
lsoa_df_flat['date'] = pd.to_datetime(lsoa_df_flat['date'])
lsoa_df_flat.to_parquet(out_file_parquet, engine="pyarrow", index=False)

#### Read the parquet file back to test everything is okay

In [ ]:
%%time
lsoa_df_flat_loaded = pd.read_parquet(out_file_parquet, engine="pyarrow")
lsoa_df_flat_loaded

In [ ]:
lsoa_df_flat_loaded.dtypes

### End of script